# Lab 05 · Clustering
**Advanced Data Mining — M.Sc. Data Science** · Dr. Ali Louati · Prince Sattam Bin Abdulaziz University
Companion to **Chapter 5**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alilouati/advanced-data-mining/blob/main/labs/Lab05_Clustering.ipynb)

**In this lab you will:**
- scale data and run k-means
- choose k with the elbow method and silhouette scores
- build a dendrogram (hierarchical clustering)
- compare k-means, DBSCAN and Gaussian mixtures on different shapes

> **How to use:** run the cells from top to bottom (`Shift + Enter`). Cells marked ✏️ are for you to modify.
> Every step works with the built-in data set; set `USE_MY_CSV = True` to try your own CSV file.

In [ ]:
# Setup: libraries used in this lab (all pre-installed on Google Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (8, 4.5)
np.random.seed(42)

## 1 · Data: iris flowers (we hide the species labels)

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame.drop(columns="target")
true_labels = iris.target            # used only at the end to evaluate
df.head()

In [ ]:
# ✏️ Optional: use your own data set instead of the built-in one
USE_MY_CSV = False   # change to True, run the cell, then choose a .csv file

if USE_MY_CSV:
    from google.colab import files          # works on Google Colab
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
    print("Loaded your file:", df.shape)
else:
    print("Using the built-in data set (iris).")

In [ ]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(df.select_dtypes("number"))
sns.pairplot(df.select_dtypes("number").iloc[:, :4], corner=True, plot_kws={"s": 12}); plt.show()

## 2 · How many clusters? Elbow and silhouette

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ks = range(2, 11)
sse, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    sse.append(km.inertia_); sil.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(ks), sse, "o-"); axes[0].set_title("Elbow method (SSE)"); axes[0].set_xlabel("k")
axes[1].bar(list(ks), sil, color="teal"); axes[1].set_title("Mean silhouette"); axes[1].set_xlabel("k")
plt.show()

## 3 · k-means result, shown in 2-D with PCA

In [ ]:
from sklearn.decomposition import PCA
K = 3                                   # ✏️ choose k from the plots above
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
X2 = PCA(2).fit_transform(X)
centers2 = PCA(2).fit(X).transform(km.cluster_centers_)

plt.scatter(X2[:, 0], X2[:, 1], c=km.labels_, cmap="Set2", s=25)
plt.scatter(centers2[:, 0], centers2[:, 1], c="red", marker="X", s=200, label="centroids")
plt.title(f"k-means with k={K}"); plt.legend(); plt.show()

In [ ]:
# Profile the clusters: average value of each attribute per cluster
profile = df.select_dtypes("number").assign(cluster=km.labels_).groupby("cluster").mean().round(2)
sns.heatmap(profile, annot=True, cmap="YlGnBu"); plt.title("Cluster profiles"); plt.show()

## 4 · Hierarchical clustering and the dendrogram

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
Z = linkage(X, method="ward")            # ✏️ try "single", "complete", "average"
plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30)
plt.title("Dendrogram (Ward linkage)"); plt.ylabel("merge distance"); plt.show()
hier_labels = fcluster(Z, t=3, criterion="maxclust")

## 5 · Shapes matter: k-means vs. DBSCAN vs. Gaussian mixture

In [ ]:
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture

moons, _ = make_moons(400, noise=0.06, random_state=0)
blobs = make_blobs(400, centers=3, cluster_std=[1.0, 2.5, 0.5], random_state=170)[0]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, data in enumerate([moons, StandardScaler().fit_transform(blobs)]):
    algos = {"k-means": KMeans(n_clusters=2 if row == 0 else 3, n_init=10, random_state=0),
             "DBSCAN": DBSCAN(eps=0.2 if row == 0 else 0.3, min_samples=5),
             "Gaussian mixture": GaussianMixture(n_components=2 if row == 0 else 3, random_state=0)}
    for col, (name, algo) in enumerate(algos.items()):
        labels = algo.fit_predict(data)
        axes[row, col].scatter(data[:, 0], data[:, 1], c=labels, cmap="Set2", s=10)
        axes[row, col].set_title(name)
plt.tight_layout(); plt.show()

## 6 · Evaluate against the hidden labels (external validation)

In [ ]:
from sklearn.metrics import adjusted_rand_score
if not USE_MY_CSV:
    print("ARI k-means      :", round(adjusted_rand_score(true_labels, km.labels_), 3))
    print("ARI hierarchical :", round(adjusted_rand_score(true_labels, hier_labels), 3))
    print("ARI GMM          :", round(adjusted_rand_score(true_labels, GaussianMixture(3, random_state=0).fit_predict(X)), 3))

## ✏️ Exercises
1. Run k-means **without** scaling. How do the clusters change?
2. Compare single, complete and average linkage dendrograms.
3. For DBSCAN on the moons, find an `eps` that produces noise points (label −1).
4. Cluster your own CSV and describe each cluster in one sentence.